# Plant disease classification (PlantVillage subset)
Crops: tomato, potato, pepper (bell).

Preprocessing & augmentation pipelines for our three models

Run download_and_split_verified.py first. This file uses its CLASSES dictionary and train_rows/val_rows/test_rows.

Design notes (for the report, section 7.2):

* Preprocessing (resizing and normalising) is applied to every split.

* Augmentation is applied to the training split only and never to validation or test

* Augmentation choices are justified for leaf-disease images specifically, not generic defaults.


In [4]:

import timm
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.models import ResNet50_Weights

IMG_SIZE = 224
SEED = 42
torch.manual_seed(SEED)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


ModuleNotFoundError: No module named 'timm'

# 1. Augmentation (training split only)

Things for the report(justifications for each transform)

* RandomResizedCrop - photos are taken at different zoom or distance in the field. This simulates that.

* RandomHorizontalFlip/RandomVerticalFlip - a leaf photographed against a plain background has no "correct" orientation, so flipping doesn't create unrealistic images or change the label.

* RandomRotation(20)     - camera angle varies. kept modest so that leaves aren't cropped out of frame.

* ColorJitter (mild)     - lighting/white-balance varies between photos, but disease symptoms in this dataset are colour-based (yellowing, brown/black lesions), so jitter is kept SMALL to avoid washing out or faking those symptoms.

DELIBIRATELY NOT USED :

blur, heavy noise, CutMix/MixUp - these would obscure or blend the fine lesion detail the models need to tell diseases apart, which is the opposite of realistic for this task.S

In [ ]:

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


# 2. Prepocessing only, validation and test splits

* Deterministic: same image in, same tensor out

* Uses ImageNet stas since two of the three models are ImageNet pretrained(ResNet50, Vit-B/16 - need to make on this one cause we have to change it).

* Using the same stats for the custom CNN keeps the three-way comparisons fair (this a model requirement in section 6 )

In [ ]:

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# 3. Model-specific preprocessing (optional but recommended)

For a fair-comparison narrative, it's fine to report that the custom CNN and the "generic" eval_transform above were used for training/validation, while these two pull each pretrained model's own expected preprocessing for its final test-set evaluation, this is just standard practice and worth one sentence of justification in the report rather than an assumption one can make silently.

In [ ]:

"""
# ResNet50 - matches exactly what it was pretrained with

resnet_eval_transform = ResNet50_Weights.IMAGENET1K_V2.transforms()


# ViT-B/16 - pulls the model's own expected input size/normalisation

vit_model = timm.create_model("vit_base_patch16_224", pretrained=True)
vit_config = timm.data.resolve_data_config({}, model=vit_model)
vit_eval_transform = timm.data.create_transform(**vit_config)
"""



# 4. Build Dataset

Make sure to tun download_and_split_verified.ipynb first.

This needs its CLASSES dict and train_rows/val_rows/test_rows (each a list of (filepath, class_name) tuples) already in scope.

There's no data/train/<class>/ folder layout to point ImageFolder at -> the files stay inside PlantVillage-Dataset/raw/color/<class_name>/, so a small wrapper Dataset reads directly from those rows instead.

In [ ]:

class PlantDiseaseDataset(Dataset):
    def __init__(self, rows, class_names, transform):
        self.rows = rows
        self.class_to_idx = {c: i for i, c in enumerate(class_names)}
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        filepath, class_name = self.rows[idx]
        image = Image.open(filepath).convert("RGB")
        return self.transform(image), self.class_to_idx[class_name]


class_names = list(CLASSES)

train_ds = PlantDiseaseDataset(train_rows, class_names, transform=train_transform)
val_ds = PlantDiseaseDataset(val_rows, class_names, transform=eval_transform)
test_ds = PlantDiseaseDataset(test_rows, class_names, transform=eval_transform)

print(f"Classes ({len(class_names)}): {class_names}")
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}")



# Leakage prevention 

The train/val/test split above was built leaf-group-aware in download_and_split_verified.py (Tomato, Potato, Pepper,_bell all covered ). 